# 第1回: 前処理の理解

このノートブックでは、EIPLフレームワークで使用される前処理関数について学びます。
ロボット学習において、データの前処理は学習の安定性と精度に大きく影響します。

**学習内容:**
1. **正規化 (Normalization)** — データを指定範囲にスケーリングする
2. **tensor2numpy** — PyTorchテンソルをNumPy配列に変換する
3. **deprocess_img** — 正規化された画像を表示可能な形式に戻す

## 環境準備

まず、必要なライブラリをインポートします。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)

---
## 1. 正規化 (Min-Max Normalization)

### なぜ正規化が必要か？

ニューラルネットワークの学習では、入力データのスケールが大きく異なると勾配が不安定になり、学習がうまく進みません。
正規化によってデータを一定の範囲に収めることで、学習を安定させることができます。

### 数式

Min-Max正規化の数式は以下の通りです：

$$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \times (b_{\max} - b_{\min}) + b_{\min}
$$

ここで：
- $x_{\min}, x_{\max}$ : 入力データの範囲
- $b_{\min}, b_{\max}$ : 出力データの範囲（目標範囲）

### 実装

以下がEIPLで使用される `normalization` 関数です：

In [ ]:
def normalization(data, indataRange, outdataRange):
    """Min-Max正規化を行う関数

    Args:
        data: 正規化する入力データ (numpy array または tensor)
        indataRange: 入力データの範囲 [min, max]
        outdataRange: 出力データの目標範囲 [min, max]

    Returns:
        正規化されたデータ
    """
    # Step 1: [0, 1] の範囲に正規化
    data = (data - indataRange[0]) / (indataRange[1] - indataRange[0])
    # Step 2: 目標範囲にスケーリング
    data = data * (outdataRange[1] - outdataRange[0]) + outdataRange[0]
    return data

### 動作確認：簡単な例

まず、小さな配列で動作を確認しましょう。

In [ ]:
# 簡単な例: [0, 255] → [0, 1]
sample = np.array([0, 64, 128, 192, 255], dtype=np.float32)
result = normalization(sample, [0, 255], [0, 1])
print(f"入力:   {sample}")
print(f"出力 [0,1]:  {result}")

# [0, 255] → [-1, 1]
result2 = normalization(sample, [0, 255], [-1, 1])
print(f"出力 [-1,1]: {result2}")

# [0, 255] → [0.1, 0.9]
result3 = normalization(sample, [0, 255], [0.1, 0.9])
print(f"出力 [0.1,0.9]: {result3}")

### MNISTデータによる実例

MNISTの手書き数字画像を使って、実際に正規化を適用してみましょう。

In [ ]:
# MNISTデータの読み込み
mnist = datasets.MNIST(root='/tmp/mnist', train=True, download=True,
                       transform=transforms.ToTensor())
# 1枚の画像を取得 (元の範囲: [0, 255] → ToTensorで [0, 1] に変換済み)
img_tensor, label = mnist[0]
img = (img_tensor.squeeze().numpy() * 255).astype(np.float32)  # [0, 255] に戻す

print(f"ラベル: {label}")
print(f"画像の形状: {img.shape}")
print(f"元のピクセル値の範囲: [{img.min():.1f}, {img.max():.1f}]")

### 異なる正規化範囲の比較

3つの正規化範囲 `[0,1]`, `[-1,1]`, `[0.1, 0.9]` を可視化して比較します。

In [ ]:
# 3つの異なる範囲で正規化
norm_01 = normalization(img.copy(), [0, 255], [0, 1])
norm_neg = normalization(img.copy(), [0, 255], [-1, 1])
norm_eipl = normalization(img.copy(), [0, 255], [0.1, 0.9])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 元画像
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'元画像\n範囲: [0, 255]', fontsize=12)

# [0, 1]
axes[1].imshow(norm_01, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'[0, 1] 正規化\n範囲: [{norm_01.min():.2f}, {norm_01.max():.2f}]', fontsize=12)

# [-1, 1]
axes[2].imshow(norm_neg, cmap='gray', vmin=-1, vmax=1)
axes[2].set_title(f'[-1, 1] 正規化\n範囲: [{norm_neg.min():.2f}, {norm_neg.max():.2f}]', fontsize=12)

# [0.1, 0.9]
axes[3].imshow(norm_eipl, cmap='gray', vmin=0.1, vmax=0.9)
axes[3].set_title(f'[0.1, 0.9] 正規化\n範囲: [{norm_eipl.min():.2f}, {norm_eipl.max():.2f}]', fontsize=12)

for ax in axes:
    ax.axis('off')
plt.suptitle('正規化範囲の比較', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### なぜ [0.1, 0.9] を使うのか？

EIPLでは `[0.1, 0.9]` の範囲がよく使われます。その理由：

- **シグモイド活性化関数との相性**: シグモイドの出力は (0, 1) の範囲で、0や1に完全に到達しません。
  `[0.1, 0.9]` に正規化することで、ネットワークが出力しやすい範囲に収まります。
- **勾配消失の回避**: シグモイドは0や1の近くで勾配がほぼ0になります。
  端の値を避けることで、学習時の勾配消失を軽減できます。
- **数値的安定性**: 厳密な0や1を避けることで、対数計算などでの数値エラーを防ぎます。

---
## 2. tensor2numpy — テンソルからNumPy配列への変換

### なぜ専用の変換関数が必要か？

PyTorchのテンソルをNumPy配列に変換する際、以下の点に注意が必要です：

1. **計算グラフからの切り離し (`detach()`)**: `requires_grad=True` のテンソルは計算グラフに組み込まれており、直接NumPyに変換できません。
2. **GPU→CPU転送 (`cpu()`)**: GPUテンソルは直接NumPyに変換できません。まずCPUに転送する必要があります。

### 実装

In [ ]:
def tensor2numpy(x):
    """PyTorchテンソルをNumPy配列に変換する関数

    Args:
        x: PyTorchテンソル (CPU or GPU)

    Returns:
        NumPy配列
    """
    if x.device.type == "cpu":
        return x.detach().numpy()
    else:
        return x.cpu().detach().numpy()

### `detach()` が必要な理由

PyTorchでは、勾配を追跡しているテンソルは計算グラフの一部です。
`detach()` を呼ぶことで、計算グラフから切り離された新しいテンソルを取得できます。

In [ ]:
# detach() が必要なケースのデモ
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
b = a * 2  # bは計算グラフの一部

# これはエラーになる
try:
    result = b.numpy()
except RuntimeError as e:
    print(f"エラー: {e}")

# detach() を使えば変換できる
result = b.detach().numpy()
print(f"detach後: {result}")
print(f"型: {type(result)}")

### 実例：モデル出力の変換

In [ ]:
# CPUテンソルの変換
cpu_tensor = torch.randn(3, 3)
cpu_result = tensor2numpy(cpu_tensor)
print(f"CPUテンソル → NumPy:")
print(f"  入力型: {type(cpu_tensor)}, デバイス: {cpu_tensor.device}")
print(f"  出力型: {type(cpu_result)}")
print(f"  値: {cpu_result}")
print()

# 勾配付きテンソルの変換
grad_tensor = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
output = grad_tensor ** 2  # 計算グラフに接続
grad_result = tensor2numpy(output)
print(f"勾配付きテンソル → NumPy:")
print(f"  requires_grad: {output.requires_grad}")
print(f"  出力型: {type(grad_result)}")
print(f"  値: {grad_result}")
print()

# GPUテンソルの変換（GPUがあれば）
if torch.cuda.is_available():
    gpu_tensor = torch.randn(3, 3).cuda()
    gpu_result = tensor2numpy(gpu_tensor)
    print(f"GPUテンソル → NumPy:")
    print(f"  入力デバイス: {gpu_tensor.device}")
    print(f"  出力型: {type(gpu_result)}")
else:
    print("GPU は利用できませんが、CPU上でも同じように動作します。")

---
## 3. deprocess_img — 画像の逆正規化

### 概要

ニューラルネットワークの出力は正規化された範囲（例: `[-0.9, 0.9]`）の浮動小数点数です。
これを画像として表示するには、元のピクセル値 `[0, 255]` の `uint8` 形式に戻す必要があります。

### パイプライン

```
元画像 [0,255] → 正規化 [0.1,0.9] → ネットワーク処理 → 出力 [-0.9,0.9]付近 → deprocess → 表示画像 [0,255]
```

### クリッピングの重要性

ネットワークの出力は、学習で設定した正規化範囲を超えることがあります。
例えば `[-0.9, 0.9]` の範囲で学習しても、出力が `-1.2` や `1.5` になることがあります。
これをそのまま逆正規化すると、ピクセル値が `[0, 255]` の範囲外になってしまいます。

**クリッピング**により、値を安全な範囲に制限してから逆正規化を行います。

### 実装

In [ ]:
def deprocess_img(data, vmin=-0.9, vmax=0.9):
    """正規化された画像データを表示可能な形式 [0, 255] uint8 に変換する関数

    Args:
        data: 正規化された画像データ (numpy array)
        vmin: クリッピングの下限値 (default: -0.9)
        vmax: クリッピングの上限値 (default: -0.9)

    Returns:
        [0, 255] の uint8 画像データ
    """
    # クリッピング: 範囲外の値を制限
    data[np.where(data < vmin)] = vmin
    data[np.where(data > vmax)] = vmax
    # 逆正規化: [vmin, vmax] → [0, 255]
    return normalization(data, [vmin, vmax], [0, 255]).astype(np.uint8)

### クリッピングの効果を確認

クリッピングあり・なしで逆正規化の結果を比較します。

In [ ]:
# 正規化された画像にノイズを加える（ネットワーク出力を模擬）
norm_img = normalization(img.copy(), [0, 255], [0.1, 0.9])

# ノイズを加えて範囲外の値を作る
np.random.seed(42)
noisy = norm_img + np.random.normal(0, 0.3, norm_img.shape).astype(np.float32)
print(f"ノイズ付き画像の値の範囲: [{noisy.min():.3f}, {noisy.max():.3f}]")
print(f"正規化範囲 [-0.9, 0.9] を超える値が存在します")

# クリッピングなしで逆正規化
noisy_no_clip = noisy.copy()
raw_result = normalization(noisy_no_clip, [-0.9, 0.9], [0, 255])
print(f"\nクリッピングなし → 値の範囲: [{raw_result.min():.1f}, {raw_result.max():.1f}]")
print("  → [0, 255] の範囲外の値が発生！")

# クリッピングありで逆正規化
clipped_result = deprocess_img(noisy.copy(), vmin=-0.9, vmax=0.9)
print(f"クリッピングあり → 値の範囲: [{clipped_result.min()}, {clipped_result.max()}]")
print("  → 安全に [0, 255] に収まる")

### 完全なパイプラインのデモ

画像の正規化から逆正規化までの全体の流れを確認します。

In [ ]:
# === 完全なパイプライン ===

# Step 1: 元画像の取得
original = img.copy()

# Step 2: 正規化 [0, 255] → [0.1, 0.9]
normalized = normalization(original, [0, 255], [0.1, 0.9])

# Step 3: ネットワーク処理をシミュレート（少しノイズを加える）
np.random.seed(0)
network_output = normalized + np.random.normal(0, 0.05, normalized.shape).astype(np.float32)

# Step 4: 逆正規化 → 表示画像
reconstructed = deprocess_img(network_output.copy(), vmin=0.1, vmax=0.9)

# 可視化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Step 1: 元画像\n[0, 255]', fontsize=11)

axes[1].imshow(normalized, cmap='gray', vmin=0.1, vmax=0.9)
axes[1].set_title(f'Step 2: 正規化\n[{normalized.min():.2f}, {normalized.max():.2f}]', fontsize=11)

axes[2].imshow(network_output, cmap='gray', vmin=0.1, vmax=0.9)
axes[2].set_title(f'Step 3: ネットワーク出力\n[{network_output.min():.2f}, {network_output.max():.2f}]', fontsize=11)

axes[3].imshow(reconstructed, cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'Step 4: 逆正規化\n[{reconstructed.min()}, {reconstructed.max()}]', fontsize=11)

for ax in axes:
    ax.axis('off')
plt.suptitle('前処理パイプライン全体の流れ', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 演習問題

以下の演習で、今回学んだ前処理関数を自分で実装してみましょう。

### 演習1: normalization関数の実装

`normalization` 関数を自分で実装し、MNIST画像を `[0.1, 0.9]` の範囲に正規化してください。
正規化前後の画像を `matplotlib` で並べて表示してください。

**ヒント:**
- まず `[0, 1]` に正規化し、次に目標範囲にスケーリングする2段階で考える
- `plt.subplot()` で複数の画像を並べて表示できる

In [ ]:
def my_normalization(data, indataRange, outdataRange):
    # ここにコードを書いてください
    pass

# MNISTデータの読み込みと正規化
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_normalization(data, indataRange, outdataRange):
    data = (data - indataRange[0]) / (indataRange[1] - indataRange[0])
    data = data * (outdataRange[1] - outdataRange[0]) + outdataRange[0]
    return data

# MNISTから画像を取得
mnist = datasets.MNIST(root='/tmp/mnist', train=True, download=True,
                       transform=transforms.ToTensor())
img_tensor, label = mnist[0]
img = (img_tensor.squeeze().numpy() * 255).astype(np.float32)

# [0.1, 0.9] に正規化
normalized = my_normalization(img.copy(), [0, 255], [0.1, 0.9])

# 表示
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'元画像 [{img.min():.0f}, {img.max():.0f}]')
axes[0].axis('off')
axes[1].imshow(normalized, cmap='gray', vmin=0.1, vmax=0.9)
axes[1].set_title(f'正規化後 [{normalized.min():.2f}, {normalized.max():.2f}]')
axes[1].axis('off')
plt.tight_layout()
plt.show()
```

</details>

### 演習2: tensor2numpy関数の実装

`tensor2numpy` 関数を自分で実装してください。
以下の3つのケースで動作を確認してください：
1. 通常のCPUテンソル
2. `requires_grad=True` のテンソルから作られた計算結果
3. GPUテンソル（GPUがなければスキップ可）

**ヒント:**
- `x.device.type` でデバイスを確認できる
- `detach()` で計算グラフから切り離す
- `cpu()` でGPUからCPUに転送する

In [ ]:
def my_tensor2numpy(x):
    # ここにコードを書いてください
    pass

# テスト
# ケース1: 通常のCPUテンソル
t1 = torch.randn(3, 3)
# ここにコードを書いてください

# ケース2: 勾配付きテンソル
t2 = torch.tensor([1.0, 2.0], requires_grad=True)
t2_out = t2 * 3
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_tensor2numpy(x):
    if x.device.type == "cpu":
        return x.detach().numpy()
    else:
        return x.cpu().detach().numpy()

# ケース1: 通常のCPUテンソル
t1 = torch.randn(3, 3)
r1 = my_tensor2numpy(t1)
print(f"ケース1 - CPUテンソル: {type(r1)}, shape={r1.shape}")

# ケース2: 勾配付きテンソル
t2 = torch.tensor([1.0, 2.0], requires_grad=True)
t2_out = t2 * 3
r2 = my_tensor2numpy(t2_out)
print(f"ケース2 - 勾配付き: {type(r2)}, 値={r2}")

# ケース3: GPUテンソル
if torch.cuda.is_available():
    t3 = torch.randn(2, 2).cuda()
    r3 = my_tensor2numpy(t3)
    print(f"ケース3 - GPUテンソル: {type(r3)}, 値={r3}")
else:
    print("ケース3 - GPU未使用（スキップ）")
```

</details>

### 演習3: deprocess_img関数の実装

`deprocess_img` 関数を自分で実装し、以下の手順で動作を確認してください：
1. MNIST画像を `[0.1, 0.9]` に正規化する
2. ノイズを加えて範囲外の値を作る（ネットワーク出力の模擬）
3. `deprocess_img` で元のピクセル値範囲 `[0, 255]` に戻す
4. 元画像と復元画像を並べて表示する

**ヒント:**
- `np.where(data < vmin)` で閾値以下のインデックスを取得できる
- 先にクリッピング、次に `normalization` で逆変換する
- `.astype(np.uint8)` で整数型に変換する

In [ ]:
def my_deprocess_img(data, vmin=0.1, vmax=0.9):
    # ここにコードを書いてください
    pass

# テスト
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_deprocess_img(data, vmin=0.1, vmax=0.9):
    data[np.where(data < vmin)] = vmin
    data[np.where(data > vmax)] = vmax
    return normalization(data, [vmin, vmax], [0, 255]).astype(np.uint8)

# MNIST画像を取得
mnist = datasets.MNIST(root='/tmp/mnist', train=True, download=True,
                       transform=transforms.ToTensor())
img_tensor, label = mnist[0]
original = (img_tensor.squeeze().numpy() * 255).astype(np.float32)

# 正規化
normalized = normalization(original.copy(), [0, 255], [0.1, 0.9])

# ノイズ追加（ネットワーク出力の模擬）
np.random.seed(42)
noisy_output = normalized + np.random.normal(0, 0.1, normalized.shape).astype(np.float32)

# 逆正規化
restored = my_deprocess_img(noisy_output.copy(), vmin=0.1, vmax=0.9)

# 表示
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('元画像')
axes[0].axis('off')
axes[1].imshow(noisy_output, cmap='gray')
axes[1].set_title(f'ノイズ付き出力\n[{noisy_output.min():.2f}, {noisy_output.max():.2f}]')
axes[1].axis('off')
axes[2].imshow(restored, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('復元画像')
axes[2].axis('off')
plt.tight_layout()
plt.show()
```

</details>

---
## まとめ

今回学んだ3つの前処理関数をまとめます：

| 関数 | 用途 | ポイント |
|------|------|----------|
| `normalization` | データのスケーリング | 入力範囲→出力範囲への線形変換 |
| `tensor2numpy` | テンソル→NumPy変換 | `detach()` と `cpu()` の使い分け |
| `deprocess_img` | 画像の逆正規化 | クリッピング後に逆変換 |

### 前処理パイプラインの全体像

```
[入力画像 uint8]
    │  normalization([0,255] → [0.1,0.9])
    ▼
[正規化画像 float]
    │  ニューラルネットワーク
    ▼
[ネットワーク出力 float]
    │  tensor2numpy (テンソル→NumPy)
    │  deprocess_img (逆正規化)
    ▼
[出力画像 uint8]
```

次回はこれらの関数を使って、実際のロボット学習データの前処理を行います。